In [ ]:
# =============================================================================
# FUMD-AI Preprocessing Workflow -- Step 6: Label cell migrations (the AI-ready dataset)
# =============================================================================
# Step:         6 of 7 (final output stage)
# Summary:      Reconstruct stable serving-cell history and label migration/destination -- produces the AI-ready dataset.
#
# Author(s):
#   - Cristina Bernad (ORCID: 0000-0001-9537-415X)
#   - Sonja Filiposka <sonja.filiposka@finki.ukim.mk> (ORCID: 0000-0003-0034-2855)
#   - Katja Gilly (ORCID: 0000-0002-8985-0639)
#
# Copyright:    (c) 2026 Cristina Bernad, Sonja Filiposka, Katja Gilly
# Repository:   https://github.com/FUMD-AI/fumd-ai-preprocessing-workflow
# Version:      1.1.4
# Funding:      This work has been funded by the FUMD-AI project, an EOSC GRAVITY -
#             Inter Project with Grant Number 25-EOSC-GRV-INTER-013.
#
# -----------------------------------------------------------------------------
# Licence
# Unless otherwise indicated:
#
#   * Source code in this notebook is licensed under the MIT License.
#
#   * Explanatory text and original figures are licensed under Creative
#     Commons Attribution 4.0 International (CC BY 4.0). Input datasets
#     retain the licences stated in their corresponding metadata or
#     source records.
#
# SPDX-License-Identifier: MIT
# -----------------------------------------------------------------------------
#
# Structured, machine-readable metadata for this workflow (authors, license,
# inputs/outputs per step) is also maintained in ro-crate-metadata.json at
# the repository root - update both together if either changes.
# =============================================================================


# Step 6 — Label cell migrations (the AI-ready dataset)

Part of the **FUMD-AI preprocessing workflow** (step 6 of 7 - the final
output of this notebook is the dataset used to train the model).

**Purpose.** Add two columns that turn the cleaned per-timestep dataset
into a supervised-learning-ready dataset:

- **`migration`** - `0` normally; `1` on the first sample of a pre-handover
  warning window (`WINDOW_S` seconds before a confirmed handover); `2` on
  the remaining samples of that window.
- **`destination`** - the serving cell the vehicle will actually settle on.
  Inside a warning window this is the upcoming stable cell; outside a
  window it stays "frozen" to the current/most recent stable cell, so that
  brief ping-pong blips between two cells are never reported as a real
  destination.

A vehicle's raw `servingCell` trace is noisy: it briefly flickers between
two or three cells during a real handover, and can "ping-pong" back and
forth near a cell boundary without ever really migrating. Telling these
apart requires reconstructing each vehicle's *stable* serving-cell history
first (`build_trajectories`) and then classifying every serving-cell change
against that history (`annotate_migrations`) - both implemented once in
`src/fumd_workflow/labeling.py` and imported here, rather than redefined
inline. See that module's docstrings for the full case classification
(C1 normal handover, C2a delayed handover (A-B-C), C2b ping-pong (not a
real handover), C3 handover with no prior history, C4 diagnostic-only).

**Input:** `INPUT_PATH` - the sentinel-fixed dataset from Step 5.

**Outputs**, written under `OUTPUT_DIR` (default: current directory), one
set per value in `WINDOW_S_VALUES` (see Step 7 for how the original study
used multiple window sizes to pick a good warning horizon):
- `dataset_labeled_w<W>.csv` - full dataset + `migration`/`destination`
- `events_detected_w<W>.csv` - actionable handover events actually applied
- `events_all_w<W>.csv` - every detected case, for diagnostics/plots
- `trajectories.csv` (at `TRAJECTORIES_PATH`) - per-vehicle stable-cell runs
  (shared across all `W`)


In [ ]:
import os
import sys
import pandas as pd

sys.path.insert(0, "src")  # so `import fumd_workflow` finds src/fumd_workflow/
from fumd_workflow import build_trajectories, annotate_migrations


In [ ]:
# ---- Parameters ----
# Default chains onto Step 5's default OUTPUT_PATH, so this notebook runs
# out of the box straight after Step 5 using the bundled example.
INPUT_PATH = "combined_dataset_fixed.csv"   # Step 5 output (input)

# Pre-handover warning window(s) to label, in seconds. The original study
# swept several values to find a good tradeoff between warning lead time
# and label noise; keep a single value for a normal production run.
WINDOW_S_VALUES = [3.0]
TOL_S = 0.1              # tolerance: a run counts as "stable" if duration >= WINDOW_S - TOL_S

TRAJECTORIES_PATH = "trajectories.csv"

# Directory the per-window output files (dataset_labeled_w<W>.csv etc, see
# below) are written into. Defaults to the current directory, matching this
# notebook's previous behavior; an orchestrating notebook (e.g.
# run_pipeline.ipynb) overrides this to collect every step's output in one
# place.
OUTPUT_DIR = "."


## 1. Clean up the input

- Drop rows where `servingCell == 0`: this is a sentinel for "not connected
  to any cell", logged under the wrong OMNeT++ module, and is not a real
  serving cell.
- Drop the lagged position columns (`x-1..x-7`, `y-1..y-7`): they were only
  needed to fix the SUMO sentinel values in Step 5, and are not used by the
  migration-labeling logic below.


In [ ]:
data = pd.read_csv(INPUT_PATH)
print("input shape:", data.shape)

data = data[data["servingCell"] != 0]
print("after dropping servingCell == 0:", data.shape)

lag_position_cols = [c for c in data.columns if c.startswith(("x-", "y-"))]
data = data.drop(columns=lag_position_cols)
print("dropped columns:", lag_position_cols)


## 2. Reconstruct each vehicle's stable serving-cell trajectory

A "run" is a contiguous stretch of time during which a vehicle stayed on
the same serving cell. This is computed once and reused for every
`WINDOW_S` value below.


In [ ]:
trajectories = build_trajectories(data, time_col="t", veh_col="veh_id", cell_col="servingCell")
trajectories.to_csv(TRAJECTORIES_PATH, index=False)
print(f"saved {TRAJECTORIES_PATH} - {len(trajectories)} runs across {trajectories['veh_id'].nunique()} vehicles")
trajectories.head()


## 3. Label migrations for each warning window and save

In [ ]:
for window_s in WINDOW_S_VALUES:
    # this is the single call that does all the labeling work - see
    # src/fumd_workflow/labeling.py:annotate_migrations for the full
    # handover/ping-pong classification logic
    labeled, events, events_all = annotate_migrations(
        data, trajectories, window_s=window_s, tol_s=TOL_S,
    )

    # one output file set per window size, so sweeping WINDOW_S_VALUES
    # (e.g. to compare 2s/3s/4s/5s warning horizons, see Step 7) never
    # overwrites a previous run's results
    suffix = f"w{window_s:g}"
    dataset_path = os.path.join(OUTPUT_DIR, f"dataset_labeled_{suffix}.csv")
    events_path = os.path.join(OUTPUT_DIR, f"events_detected_{suffix}.csv")
    events_all_path = os.path.join(OUTPUT_DIR, f"events_all_{suffix}.csv")

    labeled.to_csv(dataset_path, index=False)   # <- the AI-ready dataset
    events.to_csv(events_path, index=False)
    events_all.to_csv(events_all_path, index=False)

    print(f"window_s={window_s}: migration label counts = {labeled['migration'].value_counts().to_dict()}, "
          f"actionable events = {len(events)}")
    print(f"  saved {dataset_path}, {events_path}, {events_all_path}")
